# Part 1: Data Extraction & Verification

**Overview:** Before we can train machine learning models or simulate the 2026-27 season, we need high-quality historical NBA data. This notebook collects all game results across two complete NBA seasons (`2024-25` and `2025-26`), validates their completeness, and saves clean files to our local project folder.

### What This Notebook Covers:
1. **Environment Setup:** Configures portable paths so the project runs on any computer.
2. **Data Extraction:** Downloads regular-season and playoff game logs for both seasons.
3. **Data Quality Checks:** Verifies that no games are missing, duplicated, or corrupted.
4. **Data Serialization:** Saves the cleaned dataset to `data/processed/games_clean.csv` for downstream feature engineering.

## 1. Imports & Setup

We import standard data science libraries (`pandas`, `numpy`, `pathlib`) and configure the project root path.

In [1]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_GAMES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'games_raw_24_26.csv'
CLEAN_GAMES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'games_clean.csv'
REFRESH_API = False

## 2. Extract Game Data (2024-25 and 2025-26)

**Why this step matters:** To build a reliable prediction system, we need a complete record of every game played over the last two NBA seasons.

Each NBA regular season has 1,230 games. Because every game involves two teams (a home team and an away team), each game produces two team-game rows (2,460 rows per season). We also extract all playoff games to analyze postseason performance.

To keep the project fast, reproducible, and offline-capable, we cache the raw downloads locally in `data/raw/`.

In [2]:
seasons = ['2024-25', '2025-26']
season_types = ['Regular Season', 'Playoffs']

if RAW_GAMES_PATH.exists() and not REFRESH_API:
    games_combined = pd.read_csv(RAW_GAMES_PATH, dtype={'GAME_ID': str})
    print(f'Loaded cached raw games from {RAW_GAMES_PATH.relative_to(PROJECT_ROOT)}')
else:
    all_games = []
    for season in seasons:
        for stype in season_types:
            gf = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                season_type_nullable=stype
            )
            df = gf.get_data_frames()[0]
            df['SEASON_LABEL'] = season
            df['SEASON_TYPE_LABEL'] = stype
            all_games.append(df)
            time.sleep(1)  # To avoid hitting the API rate limit

    games_combined = pd.concat(all_games, ignore_index=True)
    RAW_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
    games_combined.to_csv(RAW_GAMES_PATH, index=False)
    print(f'Fetched from nba_api and cached to {RAW_GAMES_PATH.relative_to(PROJECT_ROOT)}')

Loaded cached raw games from data\raw\games_raw_24_26.csv


In [3]:
games_combined.groupby(['SEASON_LABEL', 'SEASON_TYPE_LABEL']).size()

SEASON_LABEL  SEASON_TYPE_LABEL
2024-25       Playoffs              168
              Regular Season       2460
2025-26       Playoffs              170
              Regular Season       2460
dtype: int64

**Key Finding:**

- **Regular Season Completeness:** Both seasons return exactly 2,460 team-game rows (1,230 games each), confirming zero games are missing.
- **Playoffs:** The 2024-25 playoffs produced 168 team-game rows (84 games), while the 2025-26 playoffs produced 170 team-game rows (85 games). This small difference reflects natural series length variation (more series went to 6 or 7 games in 2025-26).

## 3. Data Integrity & Quality Checks

**Why this step matters:** An analytical pipeline is only as good as the data feeding it. Before calculating team stats and player values, we verify that there are no missing values, duplicate rows, or inconsistent labels.

In [4]:
# checking for null values across all columns
print("Null values:\n", games_combined.isnull().sum())

# checking for duplicate rows
print("\nDuplicate rows:", games_combined.duplicated().sum())

# confirm season type and season label columns are correct
print("\nUnique season types:", games_combined['SEASON_TYPE_LABEL'].unique())

Null values:
 SEASON_ID            0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
GAME_ID              0
GAME_DATE            0
MATCHUP              0
WL                   0
MIN                  0
PTS                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
FG3A                 0
FG3_PCT              0
FTM                  0
FTA                  0
FT_PCT               0
OREB                 0
DREB                 0
REB                  0
AST                  0
STL                  0
BLK                  0
TOV                  0
PF                   0
PLUS_MINUS           0
SEASON_LABEL         0
SEASON_TYPE_LABEL    0
dtype: int64

Duplicate rows: 0

Unique season types: <StringArray>
['Regular Season', 'Playoffs']
Length: 2, dtype: str


**Key Finding:**

- **Zero Missing Data:** All game statistics and identifiers are 100% complete.
- **Zero Duplicates:** Every game row has a unique game and team identifier.
- **Clean Categorization:** Season types are cleanly divided into `Regular Season` and `Playoffs`.

## 4. Save Cleaned Dataset

We export the verified game dataset to `data/processed/games_clean.csv`. This file serves as the clean foundation for feature engineering in Part 2.

In [5]:
# save the combined dataframe to a CSV file to data/processed/
CLEAN_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
games_combined.to_csv(CLEAN_GAMES_PATH, index=False)